# 01 · Esplorazione dei dati numerici

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrancescoRomeo02/more-than-a-hack-2027-InkSight/blob/team-solution/solution/notebooks/01_esplorazione_dati.ipynb)

Usiamo **solo** il tracking automatico (ATD) e lo Smart Tagging: niente video, niente homography, niente broadcast.

Obiettivo del notebook: capire cosa c'è davvero nei dati prima di modellare.
- quanto e *come* manca la palla (buchi brevi o lunghi?)
- quanti giocatori per squadra sono visibili
- se la velocità collettiva dei giocatori distingue le pause dal gioco
- come leggere i tag di squadra dello Smart Tagging

> Regola: qui guardiamo i dati, **non** le etichette. La ground truth pubblica si usa solo nel notebook 03 come controllo di correttezza.

In [ ]:
# --- Setup: funziona sia in locale sia su Google Colab ----------------------------
# Su Colab: monta Drive, clona la repo in /content/mtah e legge i dati da DRIVE_DIR/data.
# params.json e la cache stanno direttamente in DRIVE_DIR; il CSV della submission viene copiato li'.
import os, sys, time, shutil, subprocess
REPO_URL  = "https://github.com/FrancescoRomeo02/more-than-a-hack-2027-InkSight.git"
BRANCH    = "team-solution"
DRIVE_DIR = "/content/drive/MyDrive/mtah2027"        # <- cartella su Drive con dentro data/tactical/...
IN_COLAB  = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/mtah"
    if not os.path.exists(os.path.join(ROOT, "games.py")):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    os.chdir(ROOT)
    # Tutto cio' che deve passare da un notebook all'altro sta su Drive (ogni notebook ha il suo runtime):
    os.environ.setdefault("MTAH_DATA_DIR", os.path.join(DRIVE_DIR, "data"))           # dati della challenge
    os.environ.setdefault("MTAH_CACHE_DIR", os.path.join(DRIVE_DIR, "cache"))         # ATD parsato, Metrica, log calibrazione
    os.environ.setdefault("MTAH_PARAMS", os.path.join(DRIVE_DIR, "params.json"))      # scritto da 02, letto da 03
    # (le variabili vanno impostate PRIMA di importare `possession`)
    if not os.path.isdir(os.path.join(os.environ["MTAH_DATA_DIR"], "tactical")):
        print("ATTENZIONE: dati non trovati in", os.environ["MTAH_DATA_DIR"], "- carica la cartella data/ su Drive")
    p = os.environ["MTAH_PARAMS"]
    print("params.json su Drive:", time.ctime(os.path.getmtime(p)) if os.path.exists(p) else "NON ANCORA PRESENTE (esegui il notebook 02)")
else:
    ROOT = os.environ.get("MTAH_ROOT") or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if not os.path.exists(os.path.join(ROOT, "games.py")):      # notebook lanciato dalla root
        ROOT = os.getcwd()

def save_to_drive(path):
    """Su Colab copia un file in DRIVE_DIR, cosi' non si perde quando il runtime si resetta."""
    if IN_COLAB:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy(path, DRIVE_DIR)
        print("copiato su Drive:", os.path.join(DRIVE_DIR, os.path.basename(path)))

sys.path[:0] = [ROOT, os.path.join(ROOT, "solution"), os.path.join(ROOT, "evaluation")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)
print("ROOT =", ROOT, "| Colab" if IN_COLAB else "| locale")

In [ ]:
from possession import data, features, emissions
t = time.time()
atd = data.load_atd()            # la prima volta parsa il txt (~1 min), poi usa la cache in solution/cache/
print(f"caricato in {time.time()-t:.1f}s", atd["ball"].shape, atd["players"].shape)
f = features.compute(atd)
g = data.graded_mask()
print("frame valutati:", g.sum())

## La palla: quanto manca e in che forma

In [ ]:
pres = f["ball_present"]
print(f"palla presente sui frame valutati: {pres[g].mean():.1%}")
gaps = [(a, b) for a, b in features.runs(~pres & g)]
lens = np.array([b - a for a, b in gaps])
print(f"buchi: {len(lens)}  mediana {np.median(lens)/25:.1f}s  quota frame mancanti in buchi >5s: {lens[lens>125].sum()/lens.sum():.0%}")
plt.figure(figsize=(8,3)); plt.hist(lens/25, bins=60); plt.xlabel("durata buco (s)"); plt.ylabel("n"); plt.title("Buchi della palla"); plt.show()

I buchi sono **pochi e lunghi**: interpolare la posizione della palla serve a poco.
La domanda giusta per ogni buco è: *palla ferma, oppure il possesso continua?*

## Giocatori visibili per squadra

In [ ]:
team = atd["team"]; vis = ~np.isnan(atd["players"][:, :, 0])
for t, name in [(0, "casa (Barcellona)"), (1, "ospiti (Athletic)")]:
    print(f"{name}: media giocatori visibili {vis[g][:, team==t].sum(1).mean():.1f} su {int((team==t).sum())} slot")

## Velocità collettiva: pause vs gioco

In [ ]:
dmin = np.fmin(f["d_home"], f["d_away"])
groups = {
    "palla vicino a un giocatore (<1.5 m)": g & (dmin < 1.5),
    "palla fuori dal campo (>1 m)": g & (f["ball_outside"] > 1),
    "palla non vista": g & ~pres,
}
plt.figure(figsize=(8,3))
for k, m in groups.items():
    plt.hist(f["players_speed"][m], bins=80, range=(0, 5), density=True, histtype="step", label=k)
plt.xlabel("velocità media giocatori, media mobile 2 s (m/s)"); plt.legend(); plt.show()

## Smart Tagging

In [ ]:
tags = data.load_smart_tagging()
print(pd.Series([t["code"] for t in tags]).value_counts())
signs = emissions.tag_team_signs(tags, f)
pd.DataFrame([{"code": c, "segno": s, "accordo_con_giocatore_piu_vicino": round(a, 2)} for c, (s, a) in signs.items()])

Il segno è deciso **algoritmicamente**: +1 se il tag `Team` coincide di solito con la squadra del giocatore più vicino alla palla (quindi indica chi ha la palla), -1 se indica l'avversario (es. le fasi *DEFENDING ...*).

In [ ]:
# Una finestra di 2 minuti per vedere le feature
a, b = 30000, 33000
x = np.arange(a, b) / 25
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].plot(x, f["d_home"][a:b], label="dist. casa"); ax[0].plot(x, f["d_away"][a:b], label="dist. ospiti"); ax[0].set_ylim(0, 15); ax[0].legend()
ax[1].plot(x, f["ball_speed"][a:b]); ax[1].set_ylabel("vel. palla"); ax[1].set_ylim(0, 30)
ax[2].plot(x, f["players_speed"][a:b]); ax[2].set_ylabel("vel. giocatori"); ax[2].set_xlabel("secondi dall'inizio del video")
plt.show()